# 04 · Toy models: Gaussian random fields and their gas cells

A toy model enters the optical depth the same way as a simulation does: as **gas cells**. The gas is a set of Voronoi
cells, in which every point of space belongs to the cell of its nearest generating point, and inside a cell the gas is
uniform. `Cells` holds the generating points, their periodic box and the cell volumes, and, per absorbing species, what
the optical depth needs (review notes, Sec. GasProperties):

| field | unit | required |
|---|---|---|
| `number_density` | comoving cm$^{-3}$ | yes |
| `temperature` | K | yes |
| `velocity` | km/s (peculiar), one vector per cell | yes |
| `turbulence` | m/s ($b_{\rm turb}$) | no (zero) |
| `collisional` | GHz ($A_{\rm coll}$) | no (zero) |

This notebook builds the toy model of SkewerJAX: a Gaussian random field (`PowerLawGRF`) and its statistics (§1), and
the cells it defines (§2). Notebook 05 builds the same container from a simulation snapshot. Runtime: ~2 min.

**Requires:** nothing: a Gaussian random field and its cells.
**Independent of:** lines, cosmology.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)

In [ ]:
plot_dag(('skewers', 'samplers', 'fields',)); plt.show()

## 1 · Gaussian random fields: `PowerLawGRF`

`PowerLawGRF` draws a periodic Gaussian random field in 1, 2 or 3 dimensions, with spectrum

$$
P(k) = A_s \left(\frac{k}{k_0}\right)^{n_s - 1} .
$$

The grid is given either by $(L, N)$ or by two of $(k_{\min}, k_{\max}, N_k)$. The normalisation is that of the P1D
estimator of the review notes, $\langle \lvert\delta_k\rvert^2\rangle / V = P(k)$, so `variance()` returns the
expected $\sigma^2$. `sample(key)` is jitted, and `jax.vmap` over keys draws a batch.

In [ ]:
from skewerjax.core import PowerLawGRF
key = jax.random.PRNGKey(1)
g1 = PowerLawGRF(A_s=1.0, k_0=1.0, n_s=-0.5, L=100.0, N=1024)
g2 = PowerLawGRF(A_s=1.0, k_0=1.0, n_s=-1.5, L=(50.0, 100.0), N=(128, 256))     # anisotropic box: y = 2 x
g3 = PowerLawGRF(A_s=4.0, k_0=2.0, n_s=-1.25, k_min=(2 * np.pi / 50,) * 3, N_k=(256,) * 3)   # mode (b): N_k modes per axis
for g in (g1, g2, g3):
    print(g, f"sigma = {float(jnp.sqrt(g.variance())):.3f}")
d1, d2, d3 = g1.sample(key), g2.sample(key), g3.sample(key)
iz = d3.shape[2] // 2                                                              # the slice z = L/2
fig, axs = plt.subplots(1, 3, figsize=(24, 7.5), layout="constrained", gridspec_kw={"width_ratios": [1.3, 0.75, 1]})
axs[0].plot(g1.x_grid()[0], d1, color=EC["density"], lw=1); axs[0].set(xlabel="x [Mpc]", ylabel=r"$\delta$", title="1D, $n_s = -0.5$")
Lx, Ly = g2.L
im = axs[1].imshow(np.asarray(d2).T, origin="lower", extent=(0, Lx, 0, Ly), cmap=CMAPS["diverging"], aspect="equal")
axs[1].set(xlabel="x [Mpc]", ylabel="y [Mpc]", title="2D, box 50 × 100 Mpc")
L3 = g3.L[0]
axs[2].imshow(np.asarray(d3[:, :, iz]).T, origin="lower", extent=(0, L3, 0, L3), cmap=CMAPS["diverging"], aspect="equal")
axs[2].set(xlabel="x [Mpc]", ylabel="y [Mpc]", title=f"3D, {d3.shape[0]}³ cells, slice z = L/2 (index {iz})")
add_panel_letters(axs); plt.show()

**Isotropic statistics.** The measured power spectrum of many draws recovers the input in every dimension.
The estimator is the one that defines the normalisation:

$$
\hat P(k) = \frac{\lvert\delta_k\rvert^2}{V},
$$

where $\delta_k$ is the discrete Fourier transform times the cell volume. It is averaged over the modes of each bin of
$\lvert k\rvert$ and over the draws, and the error bars are the scatter between draws divided by the square root of
their number.

The real-space correlation function is the same information in configuration space,

$$
\hat\xi(\mathbf r) = \frac{1}{N}\sum_{\mathbf x} \delta(\mathbf x)\, \delta(\mathbf x + \mathbf r),
\qquad
\xi_{\rm in}(\mathbf r) = \frac{1}{V}\sum_{\mathbf k} P(k)\, e^{i\mathbf k\cdot\mathbf r} .
$$

- $\hat\xi$ is computed from the same transform (Wiener–Khinchin) and averaged over the grid separations of each bin
  of $\lvert r\rvert$, up to a quarter of the box.
- $\xi_{\rm in}$ is the inverse transform of the input $P(k)$ on the same grid, with the box's own modes: the exact
  correlation of the discrete periodic field, so that $\langle\hat\xi\rangle = \xi_{\rm in}$. It has no $k = 0$ mode,
  so it integrates to zero over the box and turns negative at large separations.
- At zero separation it is the variance, $\xi_{\rm in}(0) = \sigma^2$ (`variance()`).
- The error of $\hat\xi$ is its exact Gaussian covariance over the draws. The modulus squared of every mode is
  exponentially distributed, with variance $P(k)^2$, so

$$
\mathrm{Cov}\big(\hat\xi_b, \hat\xi_{b'}\big) = \frac{2}{V^2 N_{\rm draws}} \sum_{\mathbf k \neq 0} P(k)^2\, c_b(\mathbf k)\, c_{b'}(\mathbf k),
\qquad c_b(\mathbf k) = \big\langle \cos(\mathbf k\cdot\mathbf r) \big\rangle_{\mathbf r \in b} .
$$

The figure shows, per dimension: $P(\lvert k\rvert)$ and the input; the ratio minus one; $\xi(\lvert r\rvert)$ and
$\xi_{\rm in}$; their difference in units of its error.

In [ ]:
from skewerjax.utils.plotting import plot_isotropic_recovery, plot_kpar_kperp_maps, plot_pk_mu_wedges, MU_WEDGES
draws = lambda g, n, seed: np.asarray(jax.vmap(g.sample)(jax.random.split(jax.random.PRNGKey(seed), n)))
cases = {"1D, 200 draws": (g1, draws(g1, 200, 2)), "2D (50 × 100 Mpc), 40 draws": (g2, draws(g2, 40, 3)),
         "3D (512³), 4 draws": (g3, draws(g3, 4, 4))}
fig, axs, rec = plot_isotropic_recovery(cases)
add_panel_letters(axs.T.ravel())
fig.suptitle(r"PowerLawGRF, isotropic statistics: $P(|k|)$ and $\xi(|r|)$ recover the input in 1D, 2D and 3D", fontsize=22)
plt.show()
for name, r in rec.items():
    print(f"{name:28s}: P(|k|) chi2/dof = {r['chi2_dof']:.2f}, max |P / P_in - 1| = {r['max_rel']:.3f}; xi(|r|): mean pull {r['xi_mean_pull']:+.2f}, "
          f"leading mode {r['xi_lead']:.0%} of the variance, chi2/dof with the covariance {r['xi_chi2_cov']:.2f} ({r['xi_dof']} dof)")

*Reading the figure.* The $\xi$ points of one panel are not independent. Every bin is set mostly by the same few
largest-scale modes of the box, whose realised power scatters by $1/\sqrt{n}$ for $n$ modes, so the points move up or
down together: the title of each residual panel gives the share of the variance in that common mode, and the
$\chi^2$ that uses the full covariance. A shift of every point by about one standard deviation is one fluctuation of
those modes, not a bias: its sign changes with the seed, while $P(k)$, whose bins are independent, shows none.

**Anisotropic decompositions.** A skewer survey singles out one direction, the line of sight. Here it is the
last axis of the grid: $k_\parallel$ is the wavenumber along it and $k_\perp$ the one across it. The isotropic average
above is justified only if the measured power depends on $\lvert k\rvert$ alone. Two views test this in 2D and 3D.

- **The $(k_\parallel, k_\perp)$ plane.** The measured power is the colour, with the measured iso-power contours
  (solid) drawn on the input ones (dashed) at the same levels.
  - The $\chi^2$ printed below the figure is that of $(P_{\rm meas} - P_{\rm in})/\sigma_P$ per bin. Here
    $\sigma_P = P_{\rm in}/\sqrt{n\,N_{\rm draws}}$ comes from mode counting: $\lvert\delta_k\rvert^2$ of a Gaussian field
    is exponentially distributed, so the mean of $n$ independent modes has a relative error $1/\sqrt n$. $P_{\rm in}$ is
    averaged over the modes of each bin.
  - Both axes are logarithmic (one mode per bin at low $k$, logarithmic bins above), so the modes with
    $k_\parallel = 0$ or $k_\perp = 0$ are not shown.
  - The anisotropic 2D box ($L_y = 2 L_x$) has different fundamental modes along the two directions, $2\pi/100$ and
    $2\pi/50\ \mathrm{Mpc^{-1}}$, and it still follows one $P(k)$.
- **Wedges in $\mu = k_\parallel/\lvert k\rvert$.** $\mathrm{Ly}\alpha$ 3D analyses show their measurements in
  wedges of $\mu$, the cosine of the angle to the line of sight, from far from it ($0 \le \mu < 0.5$) to along it
  ($0.95 \le \mu \le 1$), with the model on the data (Gordon et al. 2023,
  [arXiv:2308.10950](https://arxiv.org/abs/2308.10950), Fig. 4, for the correlation function in $\mu = r_\parallel/r$).
  Here the same four wedges hold the power spectrum: one panel per case with every wedge in its own colour (points:
  measured; dashed: the input averaged over the modes of each $(k, \mu)$ bin), and below it the ratio minus one in the
  same colours. For an isotropic input every wedge must recover $P(k)$; a redshift-space distortion would separate
  them.

Full checks: `scripts/validation/check_summary_statistics.py`.

In [ ]:
aniso = {name: c for name, c in cases.items() if c[0].ndim > 1}          # the 2D and 3D draws
fig, axs, chi2_map = plot_kpar_kperp_maps(aniso)
add_panel_letters(axs)
fig.suptitle(r"The $(k_\parallel, k_\perp)$ plane: measured contours on the input ones", fontsize=21)
plt.show()
print({name: f"map chi2/dof = {x:.2f}" for name, x in chi2_map.items()})

In [ ]:
fig, axs, chi2_mu = plot_pk_mu_wedges(aniso)
add_panel_letters(axs.T.ravel())
fig.suptitle(r"$P(k, \mu)$ in four wedges of $\mu = k_\parallel/|k|$, one colour per wedge: every wedge of an isotropic field recovers $P(k)$",
             fontsize=21)
plt.show()
for name, x in chi2_mu.items():
    print(f"{name:28s}: chi2/dof per wedge", {f"[{a:g}, {b:g})": round(c, 2) for a, b, c in zip(MU_WEDGES[:-1], MU_WEDGES[1:], x)})

## 2 · From the field to gas cells

`grf_cells(delta, grf, n_bar, temperature=..., velocity=...)` turns one 3D realisation into `Cells`. By default the
generating points are the **nodes of the grid**, $x_j = j L/N$. The Voronoi cell of a node is the cube of side $L/N$
centred on it, so the cells are exactly the grid cells and the gas is the realisation itself, piecewise constant: an
exact Gaussian random field at the resolution of the grid. For every species

$$
n(\mathbf x) = \bar n\,\big[1 + \delta(\mathbf x_j)\big] \quad \text{inside the cell of node } j ,
$$

with no transform or clipping:

- $\bar n$ is the mean density, because $\delta$ has no $k = 0$ mode;
- the amplitude $A_s$ must keep $1 + \delta > 0$; a warning is issued otherwise, and nothing is clipped, since clipping
  would change the statistics;
- several species share one realisation (here H I and Si III, 50 times less abundant).

The temperature and the velocity of every species are given per cell, as arrays or as functions of the generating
points. Here the temperature follows a power-law temperature–density relation, $T = T_0\,(1 + \delta)^{\gamma - 1}$ with
$T_0 = 1.5\times10^4$ K and $\gamma = 1.6$, and the velocity is a shear flow along $y$, $v_y = 60 \sin(2\pi x/L)$ km/s.
Both are functions of the generating points (the temperature through $\delta$ interpolated at them, which at the nodes
is the node value).

In [ ]:
from skewerjax.core import grf_cells, interpolate_periodic
L4, N4 = 50.0, 64
g4 = PowerLawGRF(A_s=0.04, k_0=2.0, n_s=-1.25, L=(L4,) * 3, N=(N4,) * 3)                    # sigma_delta = 0.2
delta4 = np.asarray(g4.sample(jax.random.PRNGKey(0)))
T_of = lambda x: 1.5e4 * (1.0 + np.asarray(interpolate_periodic(jnp.asarray(delta4), jnp.asarray(x), L4))) ** 0.6   # T_0 (1 + delta)^(gamma - 1)
shear = lambda x: np.stack([0 * x[:, 0], 60.0 * np.sin(2 * np.pi * x[:, 0] / L4), 0 * x[:, 0]], -1)                  # v_y(x) [km/s]
n_bar4 = {"HI": 1e-12, "SiIII": 1e-12 / 50}                                                                        # [comoving cm^-3]
cells = grf_cells(delta4, g4, n_bar4, temperature=T_of, velocity=shear)
print(cells)
print(f"cell size L/N = {cells.size_cMpc[0]:.4f} cMpc; <n_HI> = {cells.species['HI']['number_density'].mean():.4e} cm^-3 (= n_bar); "
      f"min(1 + delta) = {1 + delta4.min():.3f}; T from {cells.species['HI']['temperature'].min():.0f} to {cells.species['HI']['temperature'].max():.0f} K")

In [ ]:
from skewerjax.utils.plotting import tol_cmap
iz = N4 // 2                                                                       # the slice of nodes z = L/2
dx4 = L4 / N4
ext = (-dx4 / 2, L4 - dx4 / 2, -dx4 / 2, L4 - dx4 / 2)                             # every pixel is one cell, centred on its node
sl = lambda f: np.asarray(f).reshape(N4, N4, N4, *np.shape(f)[1:])[:, :, iz]
fig, axs = plt.subplots(1, 3, figsize=(24, 7.6), layout="constrained")
for ax, f, lab, cm in ((axs[0], np.log10(cells.species["HI"]["number_density"]), r"$\log_{10} n_{\rm HI}$ [cm$^{-3}$]", tol_cmap("iridescent")),
                       (axs[1], cells.species["HI"]["temperature"], r"$T$ [K]", tol_cmap("incandescent")),
                       (axs[2], cells.species["HI"]["velocity"][:, 1], r"$v_y$ [km/s]", CMAPS["diverging"])):
    im = ax.imshow(sl(f).T, origin="lower", extent=ext, cmap=cm, interpolation="nearest")
    fig.colorbar(im, ax=ax, label=lab, shrink=0.85)
q = sl(cells.species["HI"]["velocity"])[::4, ::4]
gx4 = np.arange(N4)[::4] * dx4
axs[2].quiver(*np.meshgrid(gx4, gx4, indexing="ij"), q[..., 0], q[..., 1], color=FG, scale=1500)
for ax, t in zip(axs, ("density (H I)", "temperature", "peculiar velocity (arrows: in-plane)")):
    ax.set(xlabel="x [cMpc]", ylabel="y [cMpc]", title=t, aspect="equal")
add_panel_letters(axs); fig.suptitle(f"The gas cells of the realisation: the slice z = L/2 of {N4}³ cells (one pixel per cell)", fontsize=21)
plt.show()

**The deposited field keeps the input spectrum.** Notebook 06 cuts every skewer into the pieces of the cells it
crosses (the deposition). Along a skewer of the grid, the deposited density averaged over each dual segment of its
points is the value of the cell that holds it: a top hat of width $\Delta = L/N$ around every node. Its P1D is
therefore the projection of the input onto the line of sight at the nodes, a discrete sum over the transverse modes
of the grid (notebook 07), which is periodic in $k_\parallel$ with period $2\pi/\Delta$, times the window of the cells:

$$
P_{\rm 1D}^{\rm cells}(k_\parallel) = \mathrm{sinc}^2\!\left(\frac{k_\parallel \Delta}{2}\right) P_{\rm 1D}\big(k_\parallel\ \mathrm{folded\ into}\ [-\pi/\Delta, \pi/\Delta)\big) .
$$

The average over each dual segment, of length $\delta y$ (the spacing of the points), adds the window
$\mathrm{sinc}^2(k_\parallel \delta y/2)$, a 3 % effect at the highest $k_\parallel$ shown.

Below, $16 \times 16$ skewers along $y$ through every fourth column of nodes, 1024 points each, are deposited with the
package's own selection and deposition (`deposit`), and their overdensity is measured with the P1D estimator of
notebook 07. The error bars are the exact scatter of the mean over these skewers for one realisation
(`tutorial.p1d_expected`).

**Random generating points.** The generating points can be anything, for example random points, as in a moving
mesh (`generators=`). Each cell then takes the realisation interpolated (trilinearly) at its generating point, and its
volume is that of its own Voronoi cell (`voronoi_volumes`, a Monte Carlo estimate), so its size $V^{1/3}$ is defined as
for a simulation cell. This smooths the field twice: the interpolation, a window $\mathrm{sinc}^2(k_i \Delta/2)$ per
axis, and the random cells, which average the field over irregular volumes of the mean size $\Delta$ and scatter
their boundaries. The second panel below measures that smoothing with the same skewers and the same estimator.

In [ ]:
from skewerjax.core import deposit, segment_fields, line_elements
from skewerjax.utils import tutorial
step4, n_pts = 4, 1024
cols4 = np.arange(0, N4, step4) * dx4                                                 # every 4th column of nodes
X4, Z4 = np.meshgrid(cols4, cols4, indexing="ij")
ys4 = np.arange(n_pts) * (L4 / n_pts)
pts4 = np.stack([np.repeat(X4.reshape(-1, 1), n_pts, 1), np.broadcast_to(ys4, (X4.size, n_pts)), np.repeat(Z4.reshape(-1, 1), n_pts, 1)], -1)
w4 = np.asarray(line_elements(jnp.asarray(pts4)))                                     # dual-segment lengths

def deposited_overdensity(c):                      # selection and deposition; the H I density of every dual segment
    used, seg = deposit(c, pts4)
    n_seg = segment_fields(seg, used.species["HI"]["number_density"], used.species["HI"]["temperature"], 0.0)["column_length"] / w4
    return n_seg / n_seg.mean() - 1.0

def p1d(d, dy):                                    # the mean periodogram of the skewers (notebook 07's normalisation)
    k = 2 * np.pi * np.fft.rfftfreq(d.shape[-1], dy)
    return k, (np.abs(np.fft.rfft(d, axis=-1)) ** 2 * dy / d.shape[-1]).mean(0)

dy4 = L4 / n_pts
rng4 = np.random.default_rng(1)
gen_r = rng4.uniform(0.0, L4, (N4 ** 3, 3))                                            # as many generators as grid cells, at random
cells_r = grf_cells(delta4, g4, n_bar4, generators=gen_r, temperature=T_of, velocity=shear)
dy4 = L4 / n_pts
d_grid, d_rand = deposited_overdensity(cells), deposited_overdensity(cells_r)
k4, P_grid = p1d(d_grid, dy4)
_, P_rand = p1d(d_rand, dy4)
sel4 = (k4 > 0) & (k4 <= 3 * np.pi / dx4) & (tutorial.cell_window(k4, dx4) > 1e-3)     # up to 3 x the grid Nyquist, not at the window's zeros
W_seg = tutorial.cell_window(k4[sel4], dy4)                                          # the average over each dual segment
E4, S4 = (x * W_seg for x in tutorial.p1d_expected(g4, k4[sel4], axis=1, window="cells", return_sigma=True, step=step4))
E4_nodes = tutorial.p1d_expected(g4, k4[sel4], axis=1, step=step4)                     # the projection at the nodes, no window
print(f"random generators: {len(cells_r)} cells, sizes V^(1/3) from {cells_r.size_cMpc.min():.3f} to {cells_r.size_cMpc.max():.3f} cMpc "
      f"(mean {cells_r.size_cMpc.mean():.3f}; grid cells {dx4:.3f})")

In [ ]:
from scipy.spatial import cKDTree
fig = plt.figure(figsize=(24, 15), layout="constrained")
gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 1.1], width_ratios=[1, 1, 1.3])
ax_g, ax_r, ax_s = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[0, 2])
zoom = 12.5                                                                          # a 12.5 cMpc corner of the slice z = L/2
u = np.linspace(-dx4 / 2, zoom, 500)
U, W = np.meshgrid(u, u, indexing="ij")
plane = np.stack([U.ravel(), W.ravel(), np.full(U.size, iz * dx4)], -1)
for ax, c, t in ((ax_g, cells, "grid generators (default): the grid cells"), (ax_r, cells_r, "random generators, cut by the slice")):
    nearest = cKDTree(c.positions, boxsize=L4).query(np.mod(plane, L4))[1]            # the cell of every pixel: its nearest generator
    im = ax.imshow(np.log10(c.species["HI"]["number_density"][nearest]).reshape(U.shape).T, origin="lower", extent=(u[0], u[-1], u[0], u[-1]),
                   cmap=tol_cmap("iridescent"), vmin=-12.25, vmax=-11.75, interpolation="nearest")
    inside = (np.abs(c.positions[:, 2] - iz * dx4) < 0.5 * dx4) & (c.positions[:, 0] < zoom) & (c.positions[:, 1] < zoom)
    ax.scatter(c.positions[inside, 0], c.positions[inside, 1], s=6, color=FG, lw=0)
    ax.set(xlabel="x [cMpc]", ylabel="y [cMpc]", title=t, aspect="equal")
fig.colorbar(im, ax=[ax_g, ax_r], label=r"$\log_{10} n_{\rm HI}$ [cm$^{-3}$]", shrink=0.85, location="left")
j0 = 0                                                                               # one skewer: x = z = 0, along y
ax_s.plot(ys4, d_grid[j0], color=EC["HI"], lw=1.8, label="grid cells")
ax_s.plot(ys4, d_rand[j0], color=TOL["red"], lw=1.4, label="random cells")
ax_s.set(xlim=(0, zoom), xlabel="y [cMpc]", ylabel=r"deposited $\delta = n/\langle n\rangle - 1$", title="along the skewer x = z = 0")
ax_s.legend(fontsize=15)
axP, axR = fig.add_subplot(gs[1, :2]), fig.add_subplot(gs[1, 2])
kk = k4[sel4]
axP.plot(kk, kk * E4_nodes / np.pi, color=DIM, ls=":", lw=2, label="projected input at the nodes (no window)")
axP.plot(kk, kk * E4 / np.pi, color=FG, lw=2.4, label="projected input × windows of the cells and the dual segments")
axP.errorbar(kk, kk * P_grid[sel4] / np.pi, kk * S4 / np.pi, fmt="o", ms=5, capsize=2, color=EC["HI"], label="grid cells (deposited)")
axP.plot(kk, kk * P_rand[sel4] / np.pi, "s", ms=5, color=TOL["red"], label="random cells (deposited)")
axP.axvline(np.pi / dx4, color=DIM, ls="--", lw=1.2)
axP.set(xscale="log", yscale="log", xlabel=r"$k_\parallel$ [cMpc$^{-1}$]", ylabel=r"$k P_{\rm 1D}/\pi$", title="P1D of the deposited overdensity, 256 skewers")
axP.legend(fontsize=15)
far = tutorial.cell_window(kk, dx4) > 0.05                                           # the ratio away from the zeros of the window
axR.errorbar(kk[far], (P_grid[sel4] / E4)[far], (S4 / E4)[far], fmt="o", ms=5, capsize=2, color=EC["HI"], label="grid cells")
axR.plot(kk[far], (P_rand[sel4] / E4)[far], "s", ms=5, color=TOL["red"], label="random cells")
axR.axhline(1, color=FG, lw=1.2); axR.axvline(np.pi / dx4, color=DIM, ls="--", lw=1.2)
axR.set(xscale="log", ylim=(0, 1.4), xlabel=r"$k_\parallel$ [cMpc$^{-1}$]", ylabel="measured / expected for grid cells"); axR.legend(fontsize=15)
add_panel_letters([ax_g, ax_r, ax_s, axP, axR])
fig.suptitle("Grid cells keep the input spectrum (times the windows of the cells); random cells smooth it", fontsize=21)
plt.show()
below = kk <= np.pi / dx4
r4 = (P_grid[sel4] - E4) / S4
kn = np.argmin(np.abs(kk - np.pi / dx4))
print(f"grid cells: chi2/dof = {np.mean(r4[below] ** 2):.2f} over the {below.sum()} independent bins up to the grid Nyquist pi / Delta = {np.pi / dx4:.2f} cMpc^-1, "
      f"{np.mean(r4 ** 2):.2f} over all {kk.size} (the bins beyond repeat the modes below)")
print(f"random cells / expected: {np.median((P_rand[sel4] / E4)[kk < 0.3]):.3f} at k < 0.3, {(P_rand[sel4] / E4)[kn]:.3f} at the grid Nyquist")

*Reading the figure.*

- **(a), (b)** The same realisation, with its generating points (dots, those within half a cell of the slice). The grid
  cells are the squares of the grid; the random cells are irregular polyhedra, cut here by the plane.
- **(c)** Along a skewer the deposited density is a sequence of steps: of width $\Delta$ for the grid cells, irregular
  and smoother for the random ones.
- **(d), (e)** The grid cells follow the projected input times the windows at every $k$: below the grid's
  Nyquist frequency (dashed) the window is the only difference from the nodes, and above it the cells' sharp edges keep
  the images of the node spectrum, damped by the window. The random cells lose power at all but the largest scales:
  this is the smoothing of the interpolation and of the irregular cells, a property of that representation (not of the
  deposition, which is exact for any cells).

**This is the object that notebook 05 produces from a simulation snapshot** (`SimulationSkewers.to_cells`): Voronoi
cells with, per species, a comoving density, a temperature and a peculiar velocity. From here on nothing depends on
where the cells came from: notebook 06 selects the cells along the skewers, cuts the skewers into the pieces of the
cells and sums their profiles into $\tau$.